In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

In [ ]:
# PAR RATES DATA
par_rates_data = pd.DataFrame({
    'Term': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 20],
    'Unit': ['Y']*13,
    'mid_rate_pct': [4.175870, 4.077600, 4.054550, 4.041500, 4.038500, 4.043500, 4.049050, 
                     4.055355, 4.062395, 4.071001, 4.094000, 4.118880, 4.101500]
})

In [ ]:
# Bootstrap discount curve
terms = par_rates_data['Term'].values
rates = par_rates_data['mid_rate_pct'].values / 100

discount_factors = []
times_disc = []

for i, (T, rate) in enumerate(zip(terms, rates)):
    times_disc.append(T)
    if i == 0:
        df = 1 / (1 + rate * T)
    else:
        sum_df = sum(discount_factors)
        df = 1 - rate * sum_df
    discount_factors.append(df)

times_disc = np.array(times_disc)
discount_factors = np.array(discount_factors)

discount_interp = interp1d(times_disc, discount_factors, kind='linear', 
                          bounds_error=False, fill_value='extrapolate')

In [ ]:
# Display discount curve
discount_df = pd.DataFrame({
    'Tenor (Years)': times_disc,
    'Discount Factor': discount_factors,
    'Zero Rate (%)': -np.log(discount_factors) / times_disc * 100
})
print(discount_df)

In [ ]:
# Credit spread data
def convert_tenor_to_years(tenor_str):
    if tenor_str.endswith('M'):
        return int(tenor_str[:-1]) / 12
    elif tenor_str.endswith('Y'):
        return int(tenor_str[:-1])

credit_spreads = pd.DataFrame({
    'Tenor': ['6M', '1Y', '2Y', '3Y', '4Y', '5Y', '7Y', '10Y', '15Y', '20Y'],
    'Credit Spread': [0.0806, 0.1184, 0.1735, 0.2421, 0.3142, 0.3856, 0.5447, 0.6442, 0.7440, 0.7662],
})

credit_spreads['Spread'] = credit_spreads['Credit Spread'] / 100
credit_spreads['Years'] = credit_spreads['Tenor'].apply(convert_tenor_to_years)

R = 0.40

In [ ]:
# Bootstrap survival probabilities
tenors = credit_spreads['Years'].values
spreads = credit_spreads['Spread'].values

survival_probs = []
times_surv = []
freq = 4

for i, (T, spread) in enumerate(zip(tenors, spreads)):
    payment_times = np.arange(0.25, T + 0.01, 0.25)
    
    if i == 0:
        hazard = spread / (1 - R)
        S = np.exp(-hazard * T)
    else:
        if len(survival_probs) > 1:
            surv_interp = interp1d(times_surv, survival_probs, kind='linear',
                                  bounds_error=False, fill_value='extrapolate')
        else:
            surv_interp = lambda t: survival_probs[0]
        
        def cds_value(S_T):
            all_times = np.append(times_surv, T)
            all_survs = np.append(survival_probs, S_T)
            temp_interp = interp1d(all_times, all_survs, kind='linear',
                                  bounds_error=False, fill_value='extrapolate')
            
            pv01 = 0
            for t in payment_times:
                if t <= T:
                    dt = 0.25
                    df = discount_interp(t)
                    S_t = temp_interp(t) if t <= T else S_T
                    pv01 += df * S_t * dt
            
            default_leg = 0
            prev_S = 1.0
            for t in payment_times:
                if t <= T:
                    df = discount_interp(t)
                    S_t = temp_interp(t) if t <= T else S_T
                    default_prob = prev_S - S_t
                    default_leg += df * default_prob
                    prev_S = S_t
            
            return spread * pv01 - (1 - R) * default_leg
        
        S_low, S_high = 0.001, 1.0
        for _ in range(50):
            S_mid = (S_low + S_high) / 2
            val = cds_value(S_mid)
            if abs(val) < 1e-8:
                break
            if val > 0:
                S_high = S_mid
            else:
                S_low = S_mid
        S = S_mid
    
    survival_probs.append(S)
    times_surv.append(T)

times_surv = np.array(times_surv)
survival_probs = np.array(survival_probs)

survival_interp = interp1d(times_surv, survival_probs, kind='linear',
                          bounds_error=False, fill_value='extrapolate')

In [ ]:
# Display survival curve
survival_df = pd.DataFrame({
    'Tenor': credit_spreads['Tenor'],
    'Years': times_surv,
    'Credit Spread (%)': credit_spreads['Credit Spread'],
    'Survival Probability': survival_probs,
    'Default Probability': 1 - survival_probs,
    'Hazard Rate (%)': -np.log(survival_probs) / times_surv * 100
})
print(survival_df)

In [ ]:
# CDS contract details
cds_notional = 10_000_000
cds_fixed_rate = 0.01
cds_start_date = datetime(2024, 12, 31)
cds_maturity_date = datetime(2032, 12, 20)
cds_maturity_years = (cds_maturity_date - cds_start_date).days / 365.25

print(f"Notional: ${cds_notional:,.0f}")
print(f"Fixed Rate: {cds_fixed_rate*100:.2f}%")
print(f"Maturity: {cds_maturity_years:.2f} years")
print(f"Recovery Rate: {R*100:.0f}%")

In [ ]:
# Generate payment schedule
payment_dates = []
current = cds_start_date
while current < cds_maturity_date:
    current = current + timedelta(days=91.3125)
    if current <= cds_maturity_date:
        payment_dates.append(current)

payment_times = np.array([(d - cds_start_date).days / 365.25 for d in payment_dates])

In [ ]:
# Value premium leg
pv_premium = 0
dt = 0.25
for t in payment_times:
    df = discount_interp(t)
    surv = survival_interp(t)
    pv_premium += df * surv * dt

premium_leg_pv = cds_fixed_rate * cds_notional * pv_premium

In [ ]:
# Value default leg
pv_default = 0
prev_surv = 1.0
all_times = np.concatenate([[0], payment_times])

for i in range(1, len(all_times)):
    t = all_times[i]
    t_prev = all_times[i-1]
    t_mid = (t + t_prev) / 2
    
    df = discount_interp(t_mid)
    surv = survival_interp(t)
    
    default_prob = prev_surv - surv
    pv_default += df * default_prob
    prev_surv = surv

default_leg_pv = (1 - R) * cds_notional * pv_default
cds_value = default_leg_pv - premium_leg_pv

In [ ]:
# Display results
print(f"Premium Leg PV: ${premium_leg_pv:,.2f}")
print(f"Default Leg PV: ${default_leg_pv:,.2f}")
print(f"CDS Value: ${cds_value:,.2f}")

In [ ]:
# Market comparison
survival_at_maturity = survival_interp(cds_maturity_years)
spread_interp = interp1d(credit_spreads['Years'], credit_spreads['Spread'],
                        kind='linear', bounds_error=False, fill_value='extrapolate')
market_spread = spread_interp(cds_maturity_years)

print(f"Survival Probability at Maturity: {survival_at_maturity:.4f}")
print(f"Default Probability at Maturity: {1-survival_at_maturity:.4f}")
print(f"Market Spread: {market_spread*10000:.2f} bps")
print(f"CDS Fixed Rate: {cds_fixed_rate*10000:.0f} bps")
print(f"Spread Difference: {(market_spread - cds_fixed_rate)*10000:.2f} bps")

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('CDS Curve Bootstrapping and Valuation Results', fontsize=14, fontweight='bold')

ax1 = axes[0, 0]
ax1.plot(times_disc, discount_factors, 'bo-', linewidth=2, markersize=6)
ax1.grid(True, alpha=0.3)
ax1.set_xlabel('Maturity (Years)', fontweight='bold')
ax1.set_ylabel('Discount Factor', fontweight='bold')
ax1.set_title('Bootstrapped Discount Curve', fontweight='bold')

ax2 = axes[0, 1]
ax2.plot(times_surv, survival_probs, 'ro-', linewidth=2, markersize=6, label='Survival')
ax2.plot(times_surv, 1 - survival_probs, 'go--', linewidth=2, markersize=6, label='Default')
ax2.axvline(cds_maturity_years, color='purple', linestyle='--', alpha=0.7)
ax2.grid(True, alpha=0.3)
ax2.set_xlabel('Maturity (Years)', fontweight='bold')
ax2.set_ylabel('Probability', fontweight='bold')
ax2.set_title('Survival and Default Probabilities', fontweight='bold')
ax2.legend()

ax3 = axes[1, 0]
hazard_rates = -np.log(survival_probs) / times_surv * 100
ax3.plot(times_surv, credit_spreads['Credit Spread'], 'mo-', linewidth=2, markersize=6, label='Credit Spread')
ax3.plot(times_surv, hazard_rates, 'co--', linewidth=2, markersize=6, label='Hazard Rate')
ax3.axhline(cds_fixed_rate*100, color='red', linestyle='--', alpha=0.7)
ax3.grid(True, alpha=0.3)
ax3.set_xlabel('Maturity (Years)', fontweight='bold')
ax3.set_ylabel('Rate (%)', fontweight='bold')
ax3.set_title('Credit Spreads vs Hazard Rates', fontweight='bold')
ax3.legend()

ax4 = axes[1, 1]
components = ['Premium Leg', 'Default Leg', 'Net CDS Value']
values = [premium_leg_pv, default_leg_pv, cds_value]
colors = ['red', 'green', 'blue' if cds_value > 0 else 'orange']
bars = ax4.bar(components, values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax4.axhline(0, color='black', linewidth=0.8)
ax4.set_ylabel('Present Value ($)', fontweight='bold')
ax4.set_title('CDS Valuation Components', fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, values):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height, f'${val:,.0f}',
            ha='center', va='bottom' if val > 0 else 'top', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()